In [ ]:
"""
01_dataset_description_raw.py

Describe the raw dataset before any cleaning: companies, bookyears,
legal forms, accounting schemas, industries, bankruptcies and missing
values. This notebook is purely descriptive, it does not modify the data.

03_data_description_cleaned.py runs the same checks on the cleaned
dataset, so the two can be compared side by side.
"""


In [ ]:
from utils.load_data_raw import load_data_raw

df = load_data_raw()


In [ ]:
import pandas as pd
from utils.print_section import print_section

# ------------------------------------------------------------------
# Dataset size
# ------------------------------------------------------------------
print_section("Basic shape")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ------------------------------------------------------------------
# Number of companies
# ------------------------------------------------------------------
print_section("Unique companies")

print(f"Unique VAT numbers: {df['vat'].nunique():,}")

# ------------------------------------------------------------------
# Panel structure: how many bookyears does each company have?
# ------------------------------------------------------------------
print_section("Panel structure")

obs_per_company = df.groupby("vat").size()
print(obs_per_company.describe())

# ------------------------------------------------------------------
# Bookyears
# ------------------------------------------------------------------
print_section("Bookyears")
print(df["bookyear"].value_counts().sort_index())

# ------------------------------------------------------------------
# Legal forms (top 20)
# ------------------------------------------------------------------
print_section("Legal forms")
print(df["rechtsvorm"].value_counts(dropna=False).head(20))

# ------------------------------------------------------------------
# Accounting schemas
# ------------------------------------------------------------------
print_section("Schema types")
print(df["nature"].value_counts(dropna=False).sort_index())

# ------------------------------------------------------------------
# Industries (top 20)
# ------------------------------------------------------------------
print_section("Industries")
print(df["industry"].value_counts(dropna=False).head(20))

# ------------------------------------------------------------------
# Bankruptcies
# ------------------------------------------------------------------
print_section("Bankruptcies")

failed_companies = df.loc[df["jaar_van_faling"].notna(), "vat"].nunique()
total_companies = df["vat"].nunique()
fail_rate = failed_companies / total_companies * 100

print(f"Unique failed companies: {failed_companies:,}")
print(f"Failure rate: {fail_rate:.2f}%")

# ------------------------------------------------------------------
# Bankruptcy years
# ------------------------------------------------------------------
print_section("Bankruptcy years")
print(df["jaar_van_faling"].value_counts(dropna=False).sort_index())

# ------------------------------------------------------------------
# Missing value summary
# ------------------------------------------------------------------
print_section("Missing value summary")

missing = df.isna().mean().mul(100)

print(f"Columns >= 90% missing: {(missing >= 90).sum()}")
print(f"Columns >= 80% missing: {(missing >= 80).sum()}")
print(f"Columns >= 50% missing: {(missing >= 50).sum()}")

# ------------------------------------------------------------------
# Most incomplete columns
# ------------------------------------------------------------------
print_section("Most incomplete columns")
print(missing.sort_values(ascending=False).head(30))

# ------------------------------------------------------------------
# Below: check whether rechtsvorm/industry are missing at random, or
# concentrated in specific schema types (nature) or years - this
# informs whether it's safe to just drop rows/columns later on.
# ------------------------------------------------------------------
print_section("Missing rechtsvorm by nature")

rechtsvorm_missing = df.groupby("nature")["rechtsvorm"].apply(
    lambda x: round(x.isna().mean() * 100, 2)
)
print(rechtsvorm_missing)

print_section("Missing industry by nature")

industry_missing = df.groupby("nature")["industry"].apply(
    lambda x: round(x.isna().mean() * 100, 2)
)
print(industry_missing)

print_section("Missing rechtsvorm by bookyear")

rechtsvorm_missing_year = df.groupby("bookyear")["rechtsvorm"].apply(
    lambda x: round(x.isna().mean() * 100, 2)
)
print(rechtsvorm_missing_year)

print_section("Missing industry by bookyear")

industry_missing_year = df.groupby("bookyear")["industry"].apply(
    lambda x: round(x.isna().mean() * 100, 2)
)
print(industry_missing_year)

print_section("Missing rechtsvorm vs missing industry")

missing_matrix = pd.crosstab(
    df["rechtsvorm"].isna(),
    df["industry"].isna(),
    margins=True,
)
print(missing_matrix)

print_section("Nature when rechtsvorm is missing")
print(df[df["rechtsvorm"].isna()]["nature"].value_counts().sort_index())

print_section("Nature when industry is missing")
print(df[df["industry"].isna()]["nature"].value_counts().sort_index())

print_section("Missing rechtsvorm by year and nature")

table = pd.crosstab(
    df["bookyear"],
    df["nature"],
    values=df["rechtsvorm"].isna(),
    aggfunc="mean",
) * 100
print(table.round(2))

# ------------------------------------------------------------------
# A few example rows to see what these missing cases look like
# ------------------------------------------------------------------
print_section("Examples with missing rechtsvorm")

cols = ["vat", "bookyear", "nature", "industry", "hoofdnacebel"]
available_cols = [c for c in cols if c in df.columns]

print(df[df["rechtsvorm"].isna()][available_cols].head(20))

print_section("Examples with missing industry")
print(df[df["industry"].isna()][available_cols].head(20))

# ------------------------------------------------------------------
# Show the first 10 rows of the raw dataset
# ------------------------------------------------------------------
print_section("First 10 rows in raw dataset")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

df[sorted(df.columns)].head(10).T
